In [26]:
import pandas as pd
import glob
import os

import numpy as np
from sklearn.metrics import accuracy_score, average_precision_score, roc_auc_score

In [27]:
root = "/home/stu14/s16/acm7552/Capstone/results/DIRE_outputs/"
ariaRoot     = f'{root}aria'
artbenchRoot =f'{root}artbench'
results = []

for dataset in [ariaRoot, artbenchRoot]:
    # get all csvs
    path = f'{dataset}/*.csv'
    #print(path)
    files = glob.glob(path)
    #print(files)

    # Read and combine them into one DataFrame
    df = pd.concat((pd.read_csv(f) for f in files), ignore_index=True)
    results.append(df)

ariaResults, artbenchResults = results[0], results[1]


In [28]:
ariaResults

,image,class,prediction
0,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,0,0.0
1,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,0,0.0
2,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,0,0.0
3,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,0,0.0
4,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,0,0.0
...,...,...,...
65968,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,1,0.0
65969,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,1,0.0
65970,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,1,0.0
65971,/home/stu14/s16/acm7552/Capstone/data/ARIA_dat...,1,0.0


In [29]:
artbenchResults

,image,class,prediction
0,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.9997
1,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
2,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
3,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
4,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
...,...,...,...
29995,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
29996,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
29997,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000
29998,/home/stu14/s16/acm7552/Capstone/data/Real_AI_...,0,0.0000


In [40]:
def evaluate_metrics(df, threshold=0.5):
    y_true = df["class"].values
    y_prob = df["prediction"].values

    y_pred = (y_prob >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    ap = average_precision_score(y_true, y_prob)
    auc = roc_auc_score(y_true, y_prob)

    return pd.Series({
        "accuracy": acc,
        "average_precision": ap,
        "auc": auc
    })

In [41]:
evaluate_metrics(ariaResults)

accuracy             0.137496
average_precision    0.880459
auc                  0.507217
dtype: float64

In [42]:
evaluate_metrics(artbenchResults)

accuracy             0.330067
average_precision    0.663632
auc                  0.484150
dtype: float64

In [47]:
f = pd.DataFrame([
    {"dataset": "ArtBench", **evaluate_metrics(artbenchResults)},
    {"dataset": "ARIA", **evaluate_metrics(ariaResults)},  
])

print(f)

f.to_csv("DIRE_evals.csv")

    dataset  accuracy  average_precision       auc
0  ArtBench  0.330067           0.663632  0.484150
1      ARIA  0.137496           0.880459  0.507217


In [33]:
def find_best_threshold(df, metric="accuracy"):
    y_true = df["class"].values
    y_prob = df["prediction"].values

    thresholds = np.linspace(0, 1, 1001)

    best_t = 0.5
    best_score = -1

    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)

        if metric == "accuracy":
            score = accuracy_score(y_true, y_pred)

        elif metric == "f1":
            from sklearn.metrics import f1_score
            score = f1_score(y_true, y_pred)

        else:
            raise ValueError("metric must be 'accuracy' or 'f1'")

        if score > best_score:
            best_score = score
            best_t = t

    return {
        "best_threshold": best_t,
        "best_score": best_score,
        "metric": metric
    }

In [34]:
find_best_threshold(ariaResults)

{'best_threshold': 0.0, 'best_score': 0.8779652281994149, 'metric': 'accuracy'}

In [37]:
find_best_threshold(artbenchResults)

{'best_threshold': 0.0, 'best_score': 0.6666666666666666, 'metric': 'accuracy'}